---
## TUGAS MANDIRI (Dikerjakan Selama 1 Minggu)

> **Tenggat waktu:** dikumpulkan paling lambat **sebelum Pertemuan 7 dimulai**.
> **Sifat tugas:** individu.

### Konteks / Skenario

Tim data engineering platform e-commerce meminta anda membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus: transaksi (CSV), data produk (JSON), dan data ulasan pelanggan (CSV terpisah) — merepresentasikan kondisi nyata di mana data tersebar di berbagai sistem dengan format berbeda-beda.

### Menyiapkan Dataset

Jalankan cell berikut untuk menghasilkan **tiga sumber data** sekaligus.

In [3]:
import numpy as np
import pandas as pd
import json

np.random.seed(33)

# Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01","2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {"product_id": i, "nama_produk": f"Produk-{i}", "kategori": np.random.choice(kategori_list),
     "harga": int(np.random.choice([25000,50000,75000,100000,150000,250000]))}
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# Sumber 3: Data ulasan (CSV) — tidak semua transaksi memiliki ulasan (realistis)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print(f"Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


### Instruksi Pengerjaan

Buat notebook baru **`Tugas6_[NPM]_[Nama Lengkap].ipynb`**, buat `SparkSession`, lalu bangun pipeline ETL lengkap mengikuti struktur **Extract → Transform → Load**:

---

**A. EXTRACT** *(bobot 15%)*

Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.

**B. TRANSFORM — Penggabungan** *(bobot 25%)*

Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk). Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid). Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).

**C. TRANSFORM — Penanganan Data Kosong & Pengayaan** *(bobot 20%)*

- Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `salah satu join yang tepat` — isi nilai kosong tersebut dengan angka **0** menggunakan `salah satu function`, sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".
- Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu)`, **sebelum** langkah `na.fill()` di atas).

**D. LOAD** *(bobot 25%)*

Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`. Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.

**E. Insight Akhir** *(bobot 15%)*

Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**? Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?

---

### Ketentuan Pengumpulan

- Kumpulkan `Tugas6_[NPM]_[Nama Lengkap].ipynb` melalui Asprak, paling lambat **1 minggu dari hari ini, pukul 23.59 WIB**.
- Pastikan Hadoop aktif dan seluruh cell sudah dijalankan (**Run All**) sebelum dikumpulkan.

### Rubrik Penilaian

| Bagian | Kriteria | Bobot |
|---|---|---|
| A. Extract | Ketiga sumber berhasil dibaca dengan skema yang benar | 15% |
| B. Transform - Join | Kedua join (left & inner) diterapkan dengan tepat sesuai kebutuhan masing-masing | 25% |
| C. Transform - Data Quality | Missing value tertangani logis; kolom `ada_ulasan` benar | 20% |
| D. Load | Data tersimpan ke HDFS sebagai Parquet dengan partisi yang benar & terverifikasi | 25% |
| E. Insight Akhir | Analisis tepat & interpretasi bisnis relevan | 15% |


## A. EXTRACT

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import numpy as np
import pandas as pd
import os

spark = SparkSession.builder \
    .appName("TUGAS ETL") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [25]:
df_transaksi_etl = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True )
df_produk_etl = spark.read.json("tugas6_produk.json")
df_ulasan_etl = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("Data transaksi:", df_transaksi_etl.count(), "baris")
df_transaksi_etl.printSchema()
print("Data pelanggan:", df_produk_etl.count(), "baris")
df_produk_etl.printSchema()
print("Data ulasan:", df_ulasan_etl.count(), "baris")
df_ulasan_etl.printSchema()


Data transaksi: 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

Data pelanggan: 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

Data ulasan: 3500 baris
root
 |-- order_id: string (nullable = true)
 |-- rating: integer (nullable = true)



## B. TRANSFORM

In [28]:
from pyspark.sql.functions import col

# 1. Join transaksi ke ulasan dengan LEFT JOIN (karena tidak semua transaksi memiliki ulasan)
df_trx_ulasan = df_transaksi_etl.join(df_ulasan_etl, on="order_id", how="left")

# 2. Join dengan produk dengan INNER JOIN (karena setiap transaksi pasti memiliki produk yang valid)
df_gabung = df_trx_ulasan.join(df_produk_etl, on="product_id", how="inner")

# 3. Menambahkan kolom total_pendapatan (unit_terjual * harga)
df_transform_b = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

df_transform_b.show(10)


+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|    kategori|nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000|  Elektronik|  Produk-21|           50000|
|         8|     TX1|           6|2026-10-25 00:00:00|  NULL|250000|     Fashion|   Produk-8|         1500000|
|        25|     TX2|           4|2026-10-13 00:00:00|     2| 25000|  Elektronik|  Produk-25|          100000|
|         3|     TX3|           4|2026-10-18 00:00:00|     5| 75000|     Fashion|   Produk-3|          300000|
|        19|     TX4|           6|2026-10-07 00:00:00|  NULL| 75000|Rumah Tangga|  Produk-19|          450000|
|        10|     TX5|           3|2026-10-01 00:00:00|     5| 75000|  Elektronik|  Produk-10|          225000|
|

## C. TRANSFORM Penanganan Data Kosong & Pengayaan



In [29]:
from pyspark.sql.functions import when

# 1. Menambahkan kolom 'ada_ulasan' SEBELUM na.fill()
df_transform_c = df_transform_b.withColumn(
    "ada_ulasan",
    when(col("rating").isNotNull(), True).otherwise(False)
)

# 2. Mengisi nilai null pada kolom rating dengan angka 0
df_etl_final = df_transform_c.na.fill({"rating": 0})

# Menampilkan sampel data hasil transformasi
df_etl_final.select(
    "order_id", "product_id", "nama_produk", "kategori", 
    "unit_terjual", "total_pendapatan", "rating", "ada_ulasan"
).show(5)

+--------+----------+-----------+------------+------------+----------------+------+----------+
|order_id|product_id|nama_produk|    kategori|unit_terjual|total_pendapatan|rating|ada_ulasan|
+--------+----------+-----------+------------+------------+----------------+------+----------+
|     TX0|        21|  Produk-21|  Elektronik|           2|           50000|     1|      true|
|     TX1|         8|   Produk-8|     Fashion|           6|         1500000|     0|     false|
|     TX2|        25|  Produk-25|  Elektronik|           4|          100000|     2|      true|
|     TX3|         3|   Produk-3|     Fashion|           4|          300000|     5|      true|
|     TX4|        19|  Produk-19|Rumah Tangga|           6|          450000|     0|     false|
+--------+----------+-----------+------------+------------+----------------+------+----------+
only showing top 5 rows



## D. LOAD

In [36]:
# Path tujuan di HDFS
hdfs_path = "hdfs://localhost:9000/user/haris/tugas6/hasil_etl"

# 1. Menyimpan hasil akhir ke HDFS format Parquet terpartisi berdasarkan kategori
df_etl_final.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_path)

# Verifikasi
print("Pipeline ETL Load ke HDFS Selesai!")
!hdfs dfs -ls -R /user/haris/tugas6/hasil_etl

# 3. Baca kembali data dari HDFS
df_hasil_read = spark.read.parquet(hdfs_path)

# 4. Tampilkan count() sebagai bukti data tersimpan utuh
print("==================================================")
print("Jumlah total baris data yang dibaca dari HDFS:", df_hasil_read.count())
print("==================================================")

Pipeline ETL Load ke HDFS Selesai!


-rw-r--r--   3 haris supergroup          0 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - haris supergroup          0 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 haris supergroup      16366 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Elektronik/part-00000-611e13ee-e3bc-4d8a-9bbc-47dd26596fc8.c000.snappy.parquet
drwxr-xr-x   - haris supergroup          0 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 haris supergroup      11133 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Fashion/part-00000-611e13ee-e3bc-4d8a-9bbc-47dd26596fc8.c000.snappy.parquet
drwxr-xr-x   - haris supergroup          0 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 haris supergroup      10392 2026-09-29 14:31 /user/haris/tugas6/hasil_etl/kategori=Kesehatan/part-00000-611e13ee-e3bc-4d8a-9bbc-47dd26596fc8.c000.snappy.parquet
drwxr-xr-x   - haris supergroup          0 2026-09-2

## E. INSIGHT AKHIR

In [34]:
from pyspark.sql.functions import count, sum as spark_sum, round

# Menghitung persentase transaksi yang memiliki ulasan per kategori produk
df_insight = df_etl_final.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("jumlah_ulasan")
).withColumn(
    "persentase_ulasan",
    round((col("jumlah_ulasan") / col("total_transaksi")) * 100, 2)
).orderBy("persentase_ulasan")

df_insight.show()

+------------+---------------+-------------+-----------------+
|    kategori|total_transaksi|jumlah_ulasan|persentase_ulasan|
+------------+---------------+-------------+-----------------+
|     Makanan|            536|          369|            68.84|
|   Kesehatan|            961|          662|            68.89|
|     Fashion|           1035|          726|            70.14|
|Rumah Tangga|            815|          575|            70.55|
|  Elektronik|           1653|         1168|            70.66|
+------------+---------------+-------------+-----------------+



Kategori produk dengan persentase ulasan paling rendah mengindikasikan minimnya partisipasi dan feedback dari pembeli setelah transaksi selesai. Informasi ini penting bagi tim marketing untuk merancang program dorongan (push campaign), seperti pemberian voucher diskon atau reward points bagi pelanggan yang bersedia memberikan ulasan. Peningkatan jumlah ulasan sangat krusial untuk membangun social proof, yang secara langsung dapat meningkatkan tingkat kepercayaan calon pembeli baru serta mendorong angka konversi penjualan.